# Local project data quality audit

Scope: static catalog and incentive files in `src/data`. This profiles repository data only; it does not validate live offers, tax rules, image URLs, or business definitions. Run from the repository root with Python 3.

In [ ]:
import collections, glob, json, os, re

vehicle_files = sorted(glob.glob('src/data/vehicles/*.json'))
vehicles = [(os.path.basename(path), row) for path in vehicle_files for row in json.load(open(path))]
incentives = json.load(open('src/data/ev-incentives-august-4.json'))
print(f'{len(vehicles)} vehicle rows across {len(vehicle_files)} body-style files')
print(f'{len(incentives)} incentive rows')

In [ ]:
# Key coverage and duplicate business keys across body-style files
for key in ('id', 'slug'):
    grouped = collections.defaultdict(list)
    for filename, row in vehicles:
        grouped[row.get(key)].append((filename, row.get('year'), row.get('make'), row.get('model'), row.get('bodyStyle')))
    duplicates = {value: records for value, records in grouped.items() if len(records) > 1}
    print(key, 'distinct:', len(grouped), 'duplicates:', duplicates)

# Null direct-image fields are not proof of an unavailable rendered image: each row
# can also supply imageSource, which is resolved by the application.
print('vehicle direct image null:', sum(row.get('image') is None for _, row in vehicles))
print('vehicle imageSource null:', sum(row.get('imageSource') is None for _, row in vehicles))

In [ ]:
# Incentive record grain, keys, null amounts and vehicle-catalog coverage
id_counts = collections.Counter(row['id'] for row in incentives)
by_vehicle = collections.Counter(row['vehicleSlug'] for row in incentives)
vehicle_slugs = {row['slug'] for _, row in vehicles}
print('duplicate incentive IDs:', [key for key, count in id_counts.items() if count > 1])
print('rows per vehicle slug:', collections.Counter(by_vehicle.values()))
print('vehicle slugs absent from catalog:', sorted(set(by_vehicle) - vehicle_slugs))
print('null amountValue:', sum(row.get('amountValue') is None for row in incentives), '/', len(incentives))
print('source labels:', collections.Counter(row.get('sourceLabel') for row in incentives))
print('location labels:', collections.Counter(row.get('locationLabel') for row in incentives))

In [ ]:
# Static state tax table checks (this checks shape/range, not legal accuracy)
import re
tax_source = open('src/data/stateVehicleTaxes.ts').read()
tax_rows = re.findall(r"\{ code: '([A-Z]{2})', name: '([^']+)', rate: ([0-9.]+) \}", tax_source)
codes = [row[0] for row in tax_rows]
rates = [float(row[2]) for row in tax_rows]
print('rows:', len(tax_rows), 'unique codes:', len(set(codes)), 'duplicate codes:', [code for code in set(codes) if codes.count(code) > 1])
print('rates outside 0..1:', [(row[0], row[2]) for row in tax_rows if not 0 <= float(row[2]) <= 1])
print('default state:', re.search(r"code === '([A-Z]{2})'", tax_source).group(1))

## Initial readout

- Vehicle `id` is unique (237/237); `slug` is not unique (235 unique), with duplicates to inspect for `2026/Porsche/911-Cabriolet` and `2026/Audi/A3`.
- 72/237 vehicle rows have a null direct `image`; all 72 have a non-null `imageSource`, so inspect resolver outcomes before classifying these as missing at runtime.
- Incentive IDs are unique and all 23 vehicle slugs map to a catalog entry; the uniform nine-programs-per-vehicle structure suggests grain is vehicle-program. Confirm that this is the intended coverage, not a comprehensive market inventory.
- 23/207 incentive rows have null `amountValue`. The catalog includes a source label with only `August 4` and a `CSV sample market` location; add source date/year, jurisdiction, and refresh semantics before using it for current savings comparisons.
- The two reused vehicle slugs can make slug-based joins fan out. Establish a canonical vehicle key and resolve body-style classification before joining.
- `stateVehicleTaxes.ts` has 51 unique two-letter entries with rates in [0, 1]; structural checks pass, but they do not establish that rates are legally current or model all local taxes.
